In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
!pip install optuna
import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [4]:
train = pd.read_csv('train_fe.csv')
test = pd.read_csv('test_fe.csv')

In [8]:
X = train.drop(columns=['id', 'PitNextLap'])
y = train['PitNextLap']
X_test = test.drop(columns=['id'])


cat_cols = ['Driver', 'Compound', 'Race']

for col in cat_cols:
    if col in X.columns:
        all_cats = pd.concat([X[col], X_test[col]]).astype(str).dropna().unique()

        # Wymuszamy na obu zbiorach ten sam, pełny słownik kategorii
        X[col] = pd.Categorical(X[col], categories=all_cats)
        X_test[col] = pd.Categorical(X_test[col], categories=all_cats)



In [6]:
def objective(trial):
    param = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight': 10,
        'enable_categorical': True,
        'tree_method': 'hist',
        'device': 'cuda',
        'random_state': 42,
        'early_stopping_rounds': 50,
        'n_jobs': -1
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

        model = xgb.XGBClassifier(**param)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        preds = model.predict_proba(X_va)[:, 1]
        cv_scores.append(roc_auc_score(y_va, preds))

    return np.mean(cv_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("\nBest parameters found by Optuna:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"'{key}': {value},")
print(f"\nBest average ROC AUC (3-fold): {study.best_value:.5f}")



[I 2026-11-05 19:20:18,965] A new study created in memory with name: no-name-b141717b-3fad-4812-919d-4f866db836d7
/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [19:20:22] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-11-05 19:20:28,629] Trial 0 finished with value: 0.938989294213528 and parameters: {'learning_rate': 0.09618017586253122, 'max_depth': 4, 'min_child_weight': 2, 'subsample': 0.9329449218005298, 'colsample_bytree': 0.6484260472107497, 'gamma': 0.012479258795467145, 'reg_alpha': 6.269407461007094e-08, 'reg_lambda': 1.711069


Best parameters found by Optuna:
'learning_rate': 0.035017789005496035,
'max_depth': 7,
'min_child_weight': 8,
'subsample': 0.70286626938331,
'colsample_bytree': 0.6876351083211638,
'gamma': 0.20799070135578834,
'reg_alpha': 3.79564668466043e-08,
'reg_lambda': 3.8414553194472925,

Best average ROC AUC (3-fold): 0.93978


In [9]:

final_params = {
    'n_estimators': 2000,
    'learning_rate': best_params['learning_rate'],
    'max_depth': best_params['max_depth'],
    'min_child_weight': best_params['min_child_weight'],
    'subsample': best_params['subsample'],
    'colsample_bytree': best_params['colsample_bytree'],
    'gamma': best_params['gamma'],
    'reg_alpha': best_params['reg_alpha'],
    'reg_lambda': best_params['reg_lambda'],
    'scale_pos_weight': 10,
    'enable_categorical': True,
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42,
    'early_stopping_rounds': 100,
    'n_jobs': -1
}

oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf_final.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model_xgb = xgb.XGBClassifier(**final_params)

    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    val_probs = model_xgb.predict_proba(X_va)[:, 1]
    oof_preds_xgb[val_idx] = val_probs
    test_preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / skf_final.n_splits

    print(f"Fold {fold+1} XGBoost ROC AUC: {roc_auc_score(y_va, val_probs):.5f}")

total_auc_xgb = roc_auc_score(y, oof_preds_xgb)
print(f"\nFinal OOF ROC AUC (XGBoost Optuna GPU): {total_auc_xgb:.5f}")

# 4. Zapis submisji
submission_xgb = pd.read_csv('sample_submission.csv')
submission_xgb['PitNextLap'] = test_preds_xgb
submission_xgb.to_csv('xgboost_optuna_submission.csv', index=False)


Fold 1 XGBoost ROC AUC: 0.94090
Fold 2 XGBoost ROC AUC: 0.94055
Fold 3 XGBoost ROC AUC: 0.94077
Fold 4 XGBoost ROC AUC: 0.93897
Fold 5 XGBoost ROC AUC: 0.93930

Final OOF ROC AUC (XGBoost Optuna GPU): 0.94010
